In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from src.to_html import buttons, cell, heading, image, paragraph

In [2]:
save_dir = Path("../stats/data")
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[["delay_in_min", "station_name", "is_canceled", "train_type"]])
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 31,046,215 records from 2 files
Files: ['data-2025-12.parquet', 'data-2026-01.parquet']


In [3]:
# Calculate statistics for each train type
train_types = ["Alle", "ICE", "IC", "RE", "RB", "S"]
stats = {}

for train_type in train_types:
    if train_type == "Alle":
        df_filtered = df
        type_label = "Zughalte"
    else:
        df_filtered = df[df["train_type"] == train_type]
        type_label = f"{train_type}-Zughalte"

    canceled_rate = df_filtered["is_canceled"].mean() * 100
    df_not_canceled = df_filtered[~df_filtered["is_canceled"]]

    total_stops = len(df_not_canceled)
    mean_delay = df_not_canceled["delay_in_min"].mean()
    punctual_rate = (df_not_canceled["delay_in_min"] < 6).mean() * 100

    stats[train_type] = {
        "total_stops": total_stops,
        "type_label": type_label,
        "mean_delay": f"{int(mean_delay)}:{int((mean_delay % 1) * 60):02d}",
        "punctual_rate": f"{punctual_rate:.0f}%",
        "canceled_rate": f"{canceled_rate:.0f}%",
    }

# Create stats text for each train type
stats_texts = {}
for train_type, data in stats.items():
    text = f"""In den letzten zwei Monaten gab es insgesamt <span class="highlight">{data["total_stops"]:,}</span> {data["type_label"]}.
Die Züge hatten eine durchschnittliche Verspätung von <span class="highlight">{data["mean_delay"]}</span> Minuten.
Die <a href="https://www.deutschebahn.com/de/konzern/konzernprofil/zahlen_fakten/puenktlichkeitswerte-6878476#">Deutsche Bahn</a> definiert eine Verspätung von weniger als 6 Minuten als pünktlich und damit waren <span class="highlight">{data["punctual_rate"]}</span> der Halte pünktlich.
Insgesamt gab es eine Ausfallquote von <span class="highlight">{data["canceled_rate"]}</span>."""
    stats_texts[train_type] = paragraph(text)

content = cell(heading("Was ist die durchschnittliche Verspätung von allen Zügen?", level=1) + buttons(stats_texts))
display(HTML(content))

In [4]:
# Calculate delay distribution for bar chart
bins = [-np.inf, 0, 5, 10, 15, 30, 60, np.inf]
labels = ["keine Verspätung", "0-5 min", "5-10 min", "10-15 min", "15-30 min", "30-60 min", "> 60 min"]

delay_distributions = {}
for train_type in train_types:
    if train_type == "Alle":
        df_filtered = df[~df["is_canceled"]]
    else:
        df_filtered = df[(df["train_type"] == train_type) & (~df["is_canceled"])]

    delay_hist = pd.cut(df_filtered["delay_in_min"], bins=bins, labels=labels)
    delay_distributions[train_type] = (delay_hist.value_counts(normalize=True) * 100).reindex(labels)

# Create bar chart
plt.figure(figsize=(12, 6))
bar_width = 0.15
x = np.arange(len(labels))

for i, train_type in enumerate(train_types):
    plt.bar(x + i * bar_width, delay_distributions[train_type].values, bar_width, label=train_type, alpha=0.8)

plt.xlabel("Verspätung")
plt.ylabel("Anteil [%]")
plt.title("Verteilung von Verspätungen nach Zuggattung")
plt.xticks(x + bar_width * 2.5, labels, rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

plt.savefig(save_dir / "verteilung_verspaetungen.png", dpi=150, bbox_inches="tight")
plt.close()

content = cell(
    paragraph("Hier ist die Verteilung der Verspätungen im Detail, aufgeschlüsselt nach Zuggattung:")
    + image("../stats/data/verteilung_verspaetungen.png", "Verteilung von Verspätungen")
)
display(HTML(content))

In [5]:
# Calculate cumulative distribution for line chart
plt.figure(figsize=(12, 6))

for train_type in train_types:
    if train_type == "Alle":
        df_filtered = df[~df["is_canceled"]]
    else:
        df_filtered = df[(df["train_type"] == train_type) & (~df["is_canceled"])]

    delay_counts = df_filtered["delay_in_min"].value_counts().sort_index()
    cumulative = delay_counts.cumsum() / len(df_filtered) * 100

    plt.plot(cumulative.index, cumulative.values, label=train_type)

plt.xlabel("Verspätung [Minuten]")
plt.ylabel("Kumulativer Anteil [%]")
plt.title("Kumulative Verteilung der Verspätungen nach Zuggattung")
plt.xlim(-5, 60)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(10))
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()

plt.savefig(save_dir / "kumulative_verteilung.png", dpi=150, bbox_inches="tight")
plt.close()

content = cell(
    paragraph("Noch genauer ist hier die kumulative Verteilung der Verspätungen:")
    + image("../stats/data/kumulative_verteilung.png", "Kumulative Verteilung der Verspätungen")
)
display(HTML(content))

In [6]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/allgemein.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))